In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os
import random
import numpy as np

SEED = 1235
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'

print(device)

################################
#     RESTART     RUNTIME      #
################################
from ssonn.train.cosine_full_cycle import CosineFullCycleLR
from ssonn.model.expandable_linear import *
from ssonn.train.lora_train import *


from dotenv import load_dotenv

load_dotenv()

In [ ]:
hyperparams = {
    "batch_size": 64,
    "max_lr": 0.01,
    "epochs_per_cycle": 16,
    "num_cycles": 5,
    # Расширение: "threshold" — нормализованная дисперсия > threshold;
    # "top_fraction" — top int(fraction * in_features) признаков по дисперсии (как в отдельном скрипте).
    "expansion_mode": "threshold",
    "threshold": 0.5,
    "expansion_top_fraction": 0.3,
    "low_dim_factor": 10,
    "momentum": 0.9,
    "optimizer_initial_lr": 0.0,
    "weight_decay": 0.0,
}

batch_size = hyperparams["batch_size"]

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST("./data", train=False, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=g)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
torch.manual_seed(SEED)
input_dim = 28 * 28
output_dim = 10
model = SelfExpandingNetwork(input_dim, output_dim).to(device)

In [ ]:
from comet_ml import start

experiment = start(
    api_key=os.environ.get("COMET_API_KEY"),
    workspace=os.environ.get("COMET_WORKSPACE"),
    project_name="mnist-lora",
)
experiment.log_parameters(hyperparams)
experiment.set_name("mnist-expandable-lowrank")

In [ ]:
def expandable_optimizer_factory(model_instance, hp):
    return optim.SGD(
        model_instance.parameters(),
        lr=float(hp.get("optimizer_initial_lr", 0.0)),
        momentum=float(hp.get("momentum", 0.9)),
        weight_decay=float(hp.get("weight_decay", 0.0)),
    )


def expandable_scheduler_factory(optimizer, hp):
    return CosineFullCycleLR(
        optimizer,
        float(hp["max_lr"]),
        int(hp["epochs_per_cycle"]),
    )


trained_model = train_expandable_cycles(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    device=device,
    hyperparams=hyperparams,
    experiment=experiment,
    use_log_tool=True,
    optimizer_factory=expandable_optimizer_factory,
    scheduler_factory=expandable_scheduler_factory,
)

In [ ]:
experiment.end()

In [ ]:
torch.save(model, 'mnist.pt')